## 1. Importul bibliotecilor si definirea directoarelor de lucru

In aceasta etapa sunt importate bibliotecile necesare pentru prelucrarea si analiza datelor. Biblioteca `pandas` va fi utilizata pentru citirea, organizarea si transformarea fisierelor de tip CSV, iar biblioteca `numpy` va fi folosita pentru operatii numerice si gestionarea valorilor lipsa. Modulul `Path` din biblioteca `pathlib` permite definirea mai clara si mai sigura a cailor catre fisiere si directoare.

Sunt definite doua directoare principale:

* `data` - contine fisierele brute colectate de la dispozitivele Fitbit, care vor fi utilizate ca sursa de date;
* `date_prelucrate` - va contine fisierele rezultate in urma procesului de curatare, transformare si pregatire pentru analiza BI si pentru modelul AI.

Prin instructiunea `OUT_DIR.mkdir(exist_ok=True)` se verifica existenta folderului destinat datelor prelucrate. Daca acesta nu exista, este creat automat, iar daca exista deja, executia continua fara aparitia unei erori.

La final, sunt afisate directoarele utilizate pentru a verifica faptul ca proiectul foloseste corect locatiile fisierelor de intrare si de iesire.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Notebook-ul se afla in folderul "notebooks",
# iar folderul principal al proiectului este cu un nivel mai sus.
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "date_prelucrate"

OUT_DIR.mkdir(exist_ok=True)

print("Director principal proiect:", PROJECT_DIR)
print("Director date brute:", DATA_DIR)
print("Director rezultate:", OUT_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Folderul 'data' nu a fost gasit in directorul principal al proiectului."
    )

print("\nFisiere gasite in folderul data:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Director principal proiect: D:\Facultate\an_2\sem_4\AI\aiaiaiaiaaiproject
Director date brute: D:\Facultate\an_2\sem_4\AI\aiaiaiaiaaiproject\data
Director rezultate: D:\Facultate\an_2\sem_4\AI\aiaiaiaiaaiproject\date_prelucrate

Fisiere gasite in folderul data:
- dailyActivity_merged.csv
- heartrate_seconds_merged.csv
- hourlyCalories_merged.csv
- hourlySteps_merged.csv
- sleepDay_merged.csv
- weightLogInfo_merged.csv


## 2. Incarcarea seturilor de date brute

In aceasta etapa sunt incarcate fisierele CSV care contin datele colectate de la dispozitivele Fitbit. Fiecare fisier este citit cu ajutorul bibliotecii `pandas` si este salvat intr-un DataFrame separat, pentru a putea fi analizat si prelucrat ulterior.

Seturile de date utilizate sunt:

* `dailyActivity_merged.csv` - contine informatii zilnice despre activitatea utilizatorilor, precum numarul total de pasi, distantele parcurse, minutele active, minutele sedentare si caloriile consumate;
* `hourlySteps_merged.csv` - contine numarul de pasi inregistrati pentru fiecare utilizator la nivel de ora;
* `hourlyCalories_merged.csv` - contine numarul de calorii consumate pentru fiecare utilizator la nivel de ora;
* `sleepDay_merged.csv` - contine informatii despre somnul utilizatorilor, precum numarul de minute dormite si timpul total petrecut in pat;
* `heartrate_seconds_merged.csv` - contine valorile ritmului cardiac inregistrate la intervale scurte de timp;
* `weightLogInfo_merged.csv` - contine informatii despre greutate si indicele de masa corporala, disponibile doar pentru o parte dintre utilizatori.

Dupa incarcarea fisierelor, este afisata dimensiunea fiecarui DataFrame sub forma `(numar de randuri, numar de coloane)`. Aceasta verificare permite observarea volumului de date disponibil pentru fiecare categorie si confirma faptul ca fisierele au fost citite corect.

Se poate observa ca nu toate tipurile de informatii sunt disponibile in aceeasi masura pentru toti utilizatorii. De exemplu, datele despre activitate sunt mai numeroase, in timp ce datele despre greutate sau ritm cardiac sunt disponibile pentru un numar mai redus de utilizatori. Aceasta diferenta va fi tratata in etapele urmatoare prin pastrarea explicita a valorilor indisponibile, fara eliminarea informatiilor utile despre activitate.


In [2]:
activity = pd.read_csv(DATA_DIR / "dailyActivity_merged.csv")
steps_hourly = pd.read_csv(DATA_DIR / "hourlySteps_merged.csv")
calories_hourly = pd.read_csv(DATA_DIR / "hourlyCalories_merged.csv")
sleep = pd.read_csv(DATA_DIR / "sleepDay_merged.csv")
heart = pd.read_csv(DATA_DIR / "heartrate_seconds_merged.csv")
weight = pd.read_csv(DATA_DIR / "weightLogInfo_merged.csv")

print("Activitate zilnica:", activity.shape)
print("Pasi orari:", steps_hourly.shape)
print("Calorii orare:", calories_hourly.shape)
print("Somn:", sleep.shape)
print("Puls:", heart.shape)
print("Greutate:", weight.shape)

Activitate zilnica: (940, 15)
Pasi orari: (22099, 3)
Calorii orare: (22099, 3)
Somn: (413, 5)
Puls: (2483658, 3)
Greutate: (67, 8)


## 3. Conversia coloanelor de tip data si ora

In aceasta etapa, coloanele care contin date si ore sunt transformate din format text in format `datetime`. Conversia permite sortarea cronologica, gruparea datelor pe zile sau ore si asocierea corecta a informatiilor din tabele diferite.

Pentru activitatea zilnica si somn se pastreaza doar data calendaristica, iar pentru pasi, calorii, puls si greutate se pastreaza momentul complet al inregistrarii. Valorile care nu respecta formatul asteptat sunt transformate in valori lipsa, pentru a putea fi tratate ulterior.

La final, este verificata perioada acoperita de datele despre activitate: 12 aprilie 2016 - 12 mai 2016.


In [3]:
activity["ActivityDate"] = pd.to_datetime(
    activity["ActivityDate"],
    format="%m/%d/%Y",
    errors="coerce"
).dt.normalize()

steps_hourly["ActivityHour"] = pd.to_datetime(
    steps_hourly["ActivityHour"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

calories_hourly["ActivityHour"] = pd.to_datetime(
    calories_hourly["ActivityHour"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

sleep["ActivityDate"] = pd.to_datetime(
    sleep["SleepDay"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
).dt.normalize()

heart["Time"] = pd.to_datetime(
    heart["Time"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

weight["Date"] = pd.to_datetime(
    weight["Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

print("Conversia datelor a fost realizata.")
print("Interval activitate:", activity["ActivityDate"].min(), "-", activity["ActivityDate"].max())

Conversia datelor a fost realizata.
Interval activitate: 2016-04-12 00:00:00 - 2016-05-12 00:00:00


## 4. Verificarea zilelor de activitate

In aceasta etapa sunt eliminate eventualele inregistrari duplicate pentru acelasi utilizator si aceeasi zi. Zilele cu zero pasi, sedentarism complet sau un consum redus de calorii nu sunt sterse, deoarece pot oferi informatii relevante despre comportamentul utilizatorilor.

Aceste situatii sunt marcate prin coloane noi, pentru a putea fi analizate ulterior in dashboard-ul BI.


In [4]:
activity = activity.drop_duplicates(
    subset=["Id", "ActivityDate"]
).copy()

activity["HasActivityData"] = 1

activity["IsZeroStepsDay"] = (
    activity["TotalSteps"] == 0
).astype(int)

activity["IsFullSedentaryDay"] = (
    activity["SedentaryMinutes"] >= 1440
).astype(int)

activity["IsLowCaloriesDay"] = (
    activity["Calories"] < 1000
).astype(int)

print("Numar zile activitate pastrate:", len(activity))
print("Zile cu 0 pasi:", activity["IsZeroStepsDay"].sum())
print("Zile complet sedentare:", activity["IsFullSedentaryDay"].sum())
print("Zile cu sub 1000 calorii:", activity["IsLowCaloriesDay"].sum())

Numar zile activitate pastrate: 940
Zile cu 0 pasi: 77
Zile complet sedentare: 79
Zile cu sub 1000 calorii: 12


## 5. Prelucrarea datelor despre somn

In aceasta etapa sunt eliminate inregistrarile duplicate si sunt calculati indicatori utili pentru analiza somnului: minutele petrecute treaz in pat si eficienta somnului.

De asemenea, fiecare inregistrare este clasificata orientativ ca `Somn insuficient` pentru durate sub 7 ore sau `Somn adecvat` pentru durate de cel putin 7 ore. Acest prag este utilizat pentru analiza BI, fara rol de diagnostic medical.


In [5]:
sleep = sleep.drop(columns=["SleepDay"]).drop_duplicates(
    subset=["Id", "ActivityDate"]
).copy()

sleep["MinutesAwakeInBed"] = (
    sleep["TotalTimeInBed"] -
    sleep["TotalMinutesAsleep"]
)

sleep["SleepEfficiency"] = np.where(
    sleep["TotalTimeInBed"] > 0,
    sleep["TotalMinutesAsleep"] / sleep["TotalTimeInBed"] * 100,
    np.nan
)

sleep["SleepStatus"] = np.select(
    [
        sleep["TotalMinutesAsleep"] < 420,
        sleep["TotalMinutesAsleep"] >= 420
    ],
    [
        "Somn insuficient",
        "Somn adecvat"
    ],
    default="Fara date"
)

print("Inregistrari somn dupa eliminarea duplicatelor:", len(sleep))
print(sleep[["TotalMinutesAsleep", "SleepEfficiency", "SleepStatus"]].head())

Inregistrari somn dupa eliminarea duplicatelor: 410
   TotalMinutesAsleep  SleepEfficiency       SleepStatus
0                 327        94.508671  Somn insuficient
1                 384        94.348894  Somn insuficient
2                 412        93.212670  Somn insuficient
3                 340        92.643052  Somn insuficient
4                 700        98.314607      Somn adecvat


## 6. Prelucrarea datelor despre ritmul cardiac

In aceasta etapa sunt eliminate inregistrarile incomplete si sunt marcate valorile de puls potential atipice, aflate sub 40 sau peste 200 de batai pe minut.

Deoarece masuratorile pulsului sunt foarte frecvente, datele sunt agregate la nivel de ora. Pentru fiecare utilizator si fiecare ora se calculeaza pulsul mediu, minim, maxim si numarul valorilor potential atipice.

Acesti indicatori vor fi utilizati in analiza BI doar in scop orientativ, fara rol de diagnostic medical.


In [6]:
heart = heart.dropna(
    subset=["Id", "Time", "Value"]
).copy()

heart["IsPotentiallyAtypicalHeartRate"] = (
    (heart["Value"] < 40) |
    (heart["Value"] > 200)
).astype(int)

heart["ActivityHour"] = heart["Time"].dt.floor("h")

heart_hourly = heart.groupby(
    ["Id", "ActivityHour"],
    as_index=False
).agg(
    AverageHeartRate=("Value", "mean"),
    MinHeartRate=("Value", "min"),
    MaxHeartRate=("Value", "max"),
    AtypicalHeartRateSamples=("IsPotentiallyAtypicalHeartRate", "sum")
)

print("Inregistrari puls brute:", len(heart))
print("Inregistrari puls agregate pe ora:", len(heart_hourly))
print("Mostre de puls potential atipice:", heart["IsPotentiallyAtypicalHeartRate"].sum())

Inregistrari puls brute: 2483658
Inregistrari puls agregate pe ora: 6013
Mostre de puls potential atipice: 36


## 7. Crearea tabelului de activitate zilnica

In aceasta etapa, datele despre activitate sunt combinate cu datele despre somn pentru fiecare utilizator si fiecare zi. Sunt pastrate toate zilele de activitate, chiar daca pentru unele dintre ele nu exista informatii despre somn.

Tabelul rezultat este completat cu indicatori utili pentru analiza BI: existenta datelor despre somn, ziua saptamanii, identificarea weekendului, totalul minutelor active si nivelul de activitate al utilizatorului.


In [7]:
fact_daily = activity.merge(
    sleep,
    on=["Id", "ActivityDate"],
    how="left"
)

fact_daily["HasSleepData"] = (
    fact_daily["TotalMinutesAsleep"].notna()
).astype(int)

fact_daily["DayOfWeekNumber"] = (
    fact_daily["ActivityDate"].dt.dayofweek + 1
)

fact_daily["DayOfWeek"] = (
    fact_daily["ActivityDate"].dt.day_name()
)

fact_daily["IsWeekend"] = (
    fact_daily["ActivityDate"].dt.dayofweek.isin([5, 6])
).astype(int)

fact_daily["ActiveMinutes"] = (
    fact_daily["VeryActiveMinutes"] +
    fact_daily["FairlyActiveMinutes"] +
    fact_daily["LightlyActiveMinutes"]
)

fact_daily["ActivityLevel"] = pd.cut(
    fact_daily["TotalSteps"],
    bins=[-1, 4999, 9999, np.inf],
    labels=["Sedentar", "Moderat activ", "Foarte activ"]
).astype(str)

fact_daily["SleepStatus"] = (
    fact_daily["SleepStatus"].fillna("Fara date")
)

print("Fact activitate zilnica:", fact_daily.shape)
print("Utilizatori:", fact_daily["Id"].nunique())
print("Zile cu date despre somn:", fact_daily["HasSleepData"].sum())

Fact activitate zilnica: (940, 31)
Utilizatori: 33
Zile cu date despre somn: 410


## 8. Crearea tabelului de monitorizare orara

In aceasta etapa, datele despre pasi si calorii sunt combinate la nivel de utilizator si ora, apoi sunt completate cu informatiile disponibile despre ritmul cardiac.

Sunt pastrate toate inregistrarile de activitate orara, chiar daca nu exista date despre puls pentru fiecare utilizator. Tabelul rezultat include indicatori utili pentru analiza BI, precum ora zilei, ziua saptamanii, weekendul si disponibilitatea datelor despre ritmul cardiac.


In [8]:
fact_hourly = steps_hourly.merge(
    calories_hourly,
    on=["Id", "ActivityHour"],
    how="outer",
    validate="one_to_one"
)

fact_hourly = fact_hourly.merge(
    heart_hourly,
    on=["Id", "ActivityHour"],
    how="left"
)

fact_hourly["HasHeartRateData"] = (
    fact_hourly["AverageHeartRate"].notna()
).astype(int)

fact_hourly["Date"] = (
    fact_hourly["ActivityHour"].dt.normalize()
)

fact_hourly["HourOfDay"] = (
    fact_hourly["ActivityHour"].dt.hour
)

fact_hourly["DayOfWeekNumber"] = (
    fact_hourly["ActivityHour"].dt.dayofweek + 1
)

fact_hourly["DayOfWeek"] = (
    fact_hourly["ActivityHour"].dt.day_name()
)

fact_hourly["IsWeekend"] = (
    fact_hourly["ActivityHour"].dt.dayofweek.isin([5, 6])
).astype(int)

print("Fact monitorizare orara:", fact_hourly.shape)
print("Utilizatori:", fact_hourly["Id"].nunique())
print("Randuri cu puls disponibil:", fact_hourly["HasHeartRateData"].sum())

Fact monitorizare orara: (22099, 14)
Utilizatori: 33
Randuri cu puls disponibil: 6006


## 9. Crearea dimensiunii utilizator

In aceasta etapa este creat tabelul care contine toti utilizatorii identificati in seturile de date. Pentru utilizatorii care au date despre greutate, sunt calculate greutatea medie, BMI-ul mediu si categoria BMI.

De asemenea, sunt adaugati indicatori care arata daca fiecare utilizator are date disponibile despre somn, ritm cardiac sau greutate. Astfel, analiza BI poate include toti utilizatorii, chiar daca unele informatii nu sunt disponibile pentru fiecare dintre ei.


In [9]:
all_users = sorted(
    set(activity["Id"]) |
    set(steps_hourly["Id"]) |
    set(calories_hourly["Id"]) |
    set(sleep["Id"]) |
    set(heart["Id"]) |
    set(weight["Id"])
)

dim_user = pd.DataFrame({
    "Id": all_users
})

weight_profile = weight.groupby(
    "Id",
    as_index=False
).agg(
    AverageWeightKg=("WeightKg", "mean"),
    AverageBMI=("BMI", "mean")
)

weight_profile["BMICategory"] = pd.cut(
    weight_profile["AverageBMI"],
    bins=[-np.inf, 18.5, 25, 30, np.inf],
    labels=["Subponderal", "Normal", "Supraponderal", "Obezitate"],
    right=False
).astype(str)

dim_user = dim_user.merge(
    weight_profile,
    on="Id",
    how="left"
)

dim_user["HasSleepData"] = (
    dim_user["Id"].isin(sleep["Id"].unique())
).astype(int)

dim_user["HasHeartRateData"] = (
    dim_user["Id"].isin(heart["Id"].unique())
).astype(int)

dim_user["HasWeightData"] = (
    dim_user["Id"].isin(weight["Id"].unique())
).astype(int)

print("Dimensiune utilizator:", dim_user.shape)
print(dim_user.head())

Dimensiune utilizator: (33, 7)
           Id  AverageWeightKg  AverageBMI BMICategory  HasSleepData  \
0  1503960366        52.599998   22.650000      Normal             1   
1  1624580081              NaN         NaN         NaN             0   
2  1644430081              NaN         NaN         NaN             1   
3  1844505072              NaN         NaN         NaN             1   
4  1927972279       133.500000   47.540001   Obezitate             1   

   HasHeartRateData  HasWeightData  
0                 0              1  
1                 0              0  
2                 0              0  
3                 0              0  
4                 0              1  


## 10. Crearea dimensiunii de timp

In aceasta etapa este creat tabelul calendaristic pentru intreaga perioada acoperita de datele analizate.

Pentru fiecare data sunt adaugate informatii precum anul, luna, ziua saptamanii si identificarea weekendului. Acest tabel va fi utilizat in Power BI pentru filtrarea si analiza datelor in functie de timp.


In [10]:
date_min = min(
    fact_daily["ActivityDate"].min(),
    fact_hourly["Date"].min()
)

date_max = max(
    fact_daily["ActivityDate"].max(),
    fact_hourly["Date"].max()
)

dim_date = pd.DataFrame({
    "Date": pd.date_range(date_min, date_max, freq="D")
})

dim_date["Year"] = dim_date["Date"].dt.year
dim_date["MonthNumber"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.month_name()
dim_date["Day"] = dim_date["Date"].dt.day
dim_date["DayOfWeekNumber"] = dim_date["Date"].dt.dayofweek + 1
dim_date["DayOfWeek"] = dim_date["Date"].dt.day_name()
dim_date["IsWeekend"] = (
    dim_date["Date"].dt.dayofweek.isin([5, 6])
).astype(int)

print("Dimensiune data:", dim_date.shape)
print(dim_date.head())

Dimensiune data: (31, 8)
        Date  Year  MonthNumber MonthName  Day  DayOfWeekNumber  DayOfWeek  \
0 2016-04-12  2016            4     April   12                2    Tuesday   
1 2016-04-13  2016            4     April   13                3  Wednesday   
2 2016-04-14  2016            4     April   14                4   Thursday   
3 2016-04-15  2016            4     April   15                5     Friday   
4 2016-04-16  2016            4     April   16                6   Saturday   

   IsWeekend  
0          0  
1          0  
2          0  
3          0  
4          1  


## 11. Pregatirea datelor pentru modelul AI

In aceasta etapa este creat tabelul utilizat pentru modelul preantrenat de predictie a seriilor temporale. Sunt selectate datele orare despre pasi si calorii, apoi sunt ordonate cronologic pentru fiecare utilizator.

Pentru antrenarea si testarea modelului sunt pastrati doar utilizatorii care au suficiente inregistrari: minimum 512 ore de istoric si 24 de ore care vor fi prezise. Astfel, modelul va putea estima activitatea viitoare pe baza comportamentului anterior.


In [11]:
ai_hourly = fact_hourly[
    ["Id", "ActivityHour", "StepTotal", "Calories"]
].copy()

ai_hourly = ai_hourly.sort_values(
    ["Id", "ActivityHour"]
).reset_index(drop=True)

ai_hourly[["StepTotal", "Calories"]] = (
    ai_hourly[["StepTotal", "Calories"]].fillna(0)
)

series_length = ai_hourly.groupby("Id").size()

eligible_users = series_length[
    series_length >= 536
].index

ai_hourly_eligible = ai_hourly[
    ai_hourly["Id"].isin(eligible_users)
].copy()

print("Tabel AI complet:", ai_hourly.shape)
print("Utilizatori eligibili pentru TTM:", len(eligible_users))
print("Tabel AI pentru fine-tuning:", ai_hourly_eligible.shape)

Tabel AI complet: (22099, 4)
Utilizatori eligibili pentru TTM: 29
Tabel AI pentru fine-tuning: (20694, 4)


## 12. Exportul fisierelor finale

In aceasta etapa, tabelele obtinute in urma prelucrarii sunt salvate sub forma de fisiere CSV in folderul `date_prelucrate`.

Fisierele exportate vor fi utilizate pentru construirea dashboard-ului Power BI si pentru antrenarea modelului AI de predictie a activitatii.


In [12]:
fact_daily.to_csv(
    OUT_DIR / "fact_activitate_zilnica.csv",
    index=False
)

fact_hourly.to_csv(
    OUT_DIR / "fact_monitorizare_orara.csv",
    index=False
)

dim_user.to_csv(
    OUT_DIR / "dim_utilizator.csv",
    index=False
)

dim_date.to_csv(
    OUT_DIR / "dim_data.csv",
    index=False
)

ai_hourly_eligible.to_csv(
    OUT_DIR / "ai_activitate_orara.csv",
    index=False
)

print("Export finalizat cu succes.")
print()
print("Fisiere generate:")
print("- fact_activitate_zilnica.csv")
print("- fact_monitorizare_orara.csv")
print("- dim_utilizator.csv")
print("- dim_data.csv")
print("- ai_activitate_orara.csv")

Export finalizat cu succes.

Fisiere generate:
- fact_activitate_zilnica.csv
- fact_monitorizare_orara.csv
- dim_utilizator.csv
- dim_data.csv
- ai_activitate_orara.csv
